# Gridlock 2.0 -- SAM-3 vs documented RF-DETR cascade (Kaggle, T4x2 GPU)

Answers one question with data instead of opinion: **should SAM-3 (open-vocab) replace the
documented two-stage RF-DETR cascade (00_master_design.md S3.2), or stay as a gap-filler /
auto-labeler?**

Two systems are scored on the **same IDD ground truth**:

1. **RF-DETR-small screen -> RF-DETR-large confirm** -- the documented cascade. Apache-2.0,
   single multi-class forward pass per stage.
2. **SAM-3** (`facebook/sam3`, native API) -- one forward pass **per text concept per image**.
   SAM License (Nov 2025) permits commercial use (only restricts military/ITAR/weapons), so
   unlike LocateAnything-3B it is legally shippable, not just a research/auto-label tool.

**Local zero-shot RF-DETR baseline to beat (1000 IDD images, laptop CUDA):** mAP@0.5 = **0.4803**,
mAP@0.5:0.95 = **0.2828** (`results/eval_detection_20260620-184314-6b0f.md`).

**Decision rule** (applied at the bottom): if SAM-3 beats RF-DETR broadly *and* latency is
acceptable -> replace. If it only wins the COCO-gap classes (auto-rickshaw / animal / traffic
sign / vehicle-fallback) -> hybrid (SAM-3 fills gaps, RF-DETR stays the bulk/fast detector or
becomes the auto-label target). If neither -> keep RF-DETR as documented.


## 0. Install dependencies

In [ ]:
import numpy
_PINNED_NUMPY = numpy.__version__
print('numpy before install:', _PINNED_NUMPY)

# RF-DETR (+train extras), pycocotools, supervision: preinstalled torch/CUDA on Kaggle GPU images.
# Pin numpy to the pre-existing version so torchvision's compiled C-extensions don't go stale.
!pip -q install "rfdetr[train]" pycocotools supervision "numpy=={_PINNED_NUMPY}" 2>/dev/null

# SAM-3: native package (matches the validated reference-notebook API), not the HF wrapper.
!git clone -q https://github.com/facebookresearch/sam3.git /kaggle/working/sam3_repo
!pip -q install -e "/kaggle/working/sam3_repo[notebooks]" "numpy=={_PINNED_NUMPY}" 2>/dev/null
print('installed (numpy pinned to', _PINNED_NUMPY, ')')


### ⚠️ Restart the kernel now -- before running anything below

The install cell above can upgrade `numpy` underneath the **already-imported** torch/torchvision
C-extensions, which throws:

```
ValueError: numpy.dtype size changed, may indicate binary incompatibility.
```

This is the exact same reason the official SAM-3 reference notebook has a
*"Restart session before running cells below"* step -- installing `sam3[notebooks]` (and
`rfdetr`/`supervision`) touches the numpy/torch stack the same way.

**Kaggle:** session menu (top right) -> **Restart Session** (or **Run -> Restart & Run All**
from below). Do **not** re-run the install cell first -- restart, then continue from the next
cell (GPU check). Packages are already installed; nothing else needs to be re-run.


In [ ]:
import torch, time, json as _json
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {round(p.total_memory/1024**3,1)} GB')


## 1. HF token (gated repo -- never hard-code)
`facebook/sam3` requires an **approved** HF token. Add it via Kaggle: **Add-ons -> Secrets ->
`HF_TOKEN`**. Falls back to an env var if you set one in Notebook Settings instead -- either way
the token is never written into this file.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    if os.environ.get('HF_TOKEN'):
        print('HF_TOKEN loaded from environment')
    else:
        print('WARNING: no HF_TOKEN found -- add it via Kaggle Secrets before running Part B.', e)


## 2. Config -- EDIT `IDD_ROOT`
Same IDD layout as the fine-tune notebook (`JPEGImages/`, `Annotations/`, `val.txt`).

In [ ]:
from pathlib import Path

# >>> EDIT THIS to your uploaded dataset path <<<
IDD_ROOT = '/kaggle/input/idd-detection/IDD_Detection'

# Sample size -- SAM-3 does ONE forward pass per concept per image (11 concepts here), so keep
# this modest for a first pass. 250 images x 11 concepts = 2750 SAM-3 forward passes.
SUBSET = 250

OUT_DIR = Path('/kaggle/working/sam3_vs_rfdetr')
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert Path(IDD_ROOT).exists(), f'IDD_ROOT not found: {IDD_ROOT} (check /kaggle/input)'
print('val.txt:', (Path(IDD_ROOT)/'val.txt').exists())


## 3. Ground truth -- VOC -> COCO (extended categories)
Unlike the zero-shot eval done locally, GT here includes the **India domain-gap classes**
(auto-rickshaw / animal / traffic-sign / vehicle-fallback) as real scoreable categories -- SAM-3
can address them by text prompt even though RF-DETR structurally cannot (no COCO equivalent).

In [ ]:
import xml.etree.ElementTree as ET

CATS = {1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'bus', 6: 'truck',
        7: 'traffic light', 8: 'autorickshaw', 9: 'animal', 10: 'traffic sign', 11: 'vehicle fallback'}
CAT_ID = {v: k for k, v in CATS.items()}
IDD_TO_CAT = {'car': 'car', 'motorcycle': 'motorcycle', 'bus': 'bus', 'truck': 'truck',
              'bicycle': 'bicycle', 'person': 'person', 'rider': 'person',
              'traffic light': 'traffic light', 'autorickshaw': 'autorickshaw', 'animal': 'animal',
              'traffic sign': 'traffic sign', 'vehicle fallback': 'vehicle fallback'}
GAP_CATS = {'autorickshaw', 'animal', 'traffic sign', 'vehicle fallback'}

def parse_voc(xml_path):
    r = ET.parse(xml_path).getroot()
    s = r.find('size')
    w = int(float(s.findtext('width', '0'))); h = int(float(s.findtext('height', '0')))
    objs = []
    for o in r.findall('object'):
        name = (o.findtext('name') or '').strip(); b = o.find('bndbox')
        if b is None or not name:
            continue
        x1 = float(b.findtext('xmin', '0')); y1 = float(b.findtext('ymin', '0'))
        x2 = float(b.findtext('xmax', '0')); y2 = float(b.findtext('ymax', '0'))
        if x2 > x1 and y2 > y1:
            objs.append((name, x1, y1, x2, y2))
    return w, h, objs

def load_val(limit):
    entries = (Path(IDD_ROOT) / 'val.txt').read_text().split()
    out = []
    for e in entries:
        img = None
        for ext in ('.jpg', '.png'):
            p = Path(IDD_ROOT) / 'JPEGImages' / f'{e}{ext}'
            if p.exists():
                img = p
                break
        xml = Path(IDD_ROOT) / 'Annotations' / f'{e}.xml'
        if img and xml.exists():
            out.append((img, xml))
        if limit and len(out) >= limit:
            break
    return out

samples = load_val(SUBSET)
print(len(samples), 'val images loaded')

images, gt_anns, gap_counts = [], [], {c: 0 for c in GAP_CATS}
aid = 1
for img_id, (img_path, xml_path) in enumerate(samples, 1):
    w, h, objs = parse_voc(xml_path)
    images.append({'id': img_id, 'width': w, 'height': h, 'file_name': img_path.name})
    for name, x1, y1, x2, y2 in objs:
        cat = IDD_TO_CAT.get(name)
        if cat is None:
            continue
        if cat in GAP_CATS:
            gap_counts[cat] += 1
        gt_anns.append({'id': aid, 'image_id': img_id, 'category_id': CAT_ID[cat],
                         'bbox': [x1, y1, x2 - x1, y2 - y1], 'area': (x2 - x1) * (y2 - y1), 'iscrowd': 0})
        aid += 1

print(f'{len(gt_anns)} GT boxes across {len(CATS)} categories')
print('domain-gap GT counts:', gap_counts)


## 4. Part A -- documented RF-DETR cascade (screen -> confirm)
Same logic as `src/modules/detection.py::TwoStageDetector`: Stage-1 RF-DETR-small screens at a
low threshold for recall, Stage-2 RF-DETR-large confirms (same-class IoU match), adopts the
confirmed box+score.

In [ ]:
from rfdetr import RFDETRSmall, RFDETRLarge
from rfdetr.util.coco_classes import COCO_CLASSES

stage1 = RFDETRSmall()
stage2 = RFDETRLarge()
print('RF-DETR-small + RF-DETR-large loaded')

def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1, ix2, iy2 = max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0

def cascade_detect(pil_img, screen_thr=0.25, confirm_thr=0.5, confirm_iou=0.45):
    screen = stage1.predict(pil_img, threshold=screen_thr)
    confirm = stage2.predict(pil_img, threshold=confirm_thr)
    s_boxes = [(COCO_CLASSES.get(int(c), str(int(c))), tuple(b)) for b, c in zip(screen.xyxy, screen.class_id)]
    out = []
    for b, c, sc in zip(confirm.xyxy, confirm.class_id, confirm.confidence):
        name = COCO_CLASSES.get(int(c), str(int(c)))
        if any(sn == name and _iou(sb, tuple(b)) >= confirm_iou for sn, sb in s_boxes):
            out.append((name, tuple(float(v) for v in b), float(sc)))
    return out


In [ ]:
from PIL import Image

rfdetr_dts = []
t0 = time.time()
for img_id, (img_path, _xml) in enumerate(samples, 1):
    pil = Image.open(img_path).convert('RGB')
    for name, box, conf_score in cascade_detect(pil):
        if name not in CAT_ID:
            continue
        x1, y1, x2, y2 = box
        rfdetr_dts.append({'image_id': img_id, 'category_id': CAT_ID[name],
                            'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': conf_score})
    if img_id % 50 == 0:
        print(f'  RF-DETR cascade {img_id}/{len(samples)}')
rfdetr_seconds = time.time() - t0
print(f'RF-DETR cascade: {len(rfdetr_dts)} detections in {rfdetr_seconds:.1f}s '
      f'({len(samples)/rfdetr_seconds:.2f} img/s, 2 forward passes/image)')


## 5. Part B -- SAM-3 native API (per-concept text prompts)
One concept = one `set_text_prompt` call = one forward pass. 11 categories -> 11 passes/image.
This is the scalability cost being measured against RF-DETR's 2 passes/image.

In [ ]:
import torch
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

sam3_model = build_sam3_image_model(
    bpe_path='/kaggle/working/sam3_repo/sam3/assets/bpe_simple_vocab_16e6.txt.gz')
sam3_processor = Sam3Processor(sam3_model, confidence_threshold=0.25)
print('SAM-3 loaded')

CONCEPT_PROMPT = {
    'person': 'person', 'bicycle': 'bicycle', 'car': 'car', 'motorcycle': 'motorcycle',
    'bus': 'bus', 'truck': 'truck', 'traffic light': 'traffic light',
    'autorickshaw': 'auto rickshaw', 'animal': 'animal', 'traffic sign': 'traffic sign',
    'vehicle fallback': 'vehicle',
}


In [ ]:
sam3_dts = []
t0 = time.time()
for img_id, (img_path, _xml) in enumerate(samples, 1):
    pil = Image.open(img_path).convert('RGB')
    state = sam3_processor.set_image(pil)
    for cat_name, prompt in CONCEPT_PROMPT.items():
        state = sam3_processor.set_text_prompt(state=state, prompt=prompt)
        boxes = state['boxes'].to(torch.float32).cpu().numpy()
        scores = state['scores'].to(torch.float32).cpu().numpy()
        for box, conf_score in zip(boxes, scores):
            x1, y1, x2, y2 = [float(v) for v in box]
            sam3_dts.append({'image_id': img_id, 'category_id': CAT_ID[cat_name],
                              'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(conf_score)})
    if img_id % 50 == 0:
        print(f'  SAM-3 {img_id}/{len(samples)}')
sam3_seconds = time.time() - t0
n_concepts = len(CONCEPT_PROMPT)
print(f'SAM-3: {len(sam3_dts)} detections in {sam3_seconds:.1f}s '
      f'({len(samples)/sam3_seconds:.2f} img/s, {n_concepts} forward passes/image)')


## 6. Score both -- per-class AP@0.5 on identical GT
Both detection sets are scored against the SAME extended-category GT, so the gap classes are
fair game for SAM-3 (and structurally ~0 for RF-DETR -- not a bug, the documented gap).

In [ ]:
import contextlib, io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

categories = [{'id': cid, 'name': n} for n, cid in CAT_ID.items()]
coco_gt = COCO()
coco_gt.dataset = {'images': images, 'annotations': gt_anns, 'categories': categories}
with contextlib.redirect_stdout(io.StringIO()):
    coco_gt.createIndex()

def score(dts, label):
    metrics = {'mAP_50': 0.0, 'mAP_50_95': 0.0, 'per_class_AP50': {c: 0.0 for c in CAT_ID}}
    if not dts:
        return metrics
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(dts)
        ev = COCOeval(coco_gt, coco_dt, 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
    metrics['mAP_50'] = round(float(ev.stats[1]), 4)
    metrics['mAP_50_95'] = round(float(ev.stats[0]), 4)
    prec = ev.eval['precision']
    for k, name in enumerate(CAT_ID):
        p = prec[0, :, k, 0, -1]; p = p[p > -1]
        metrics['per_class_AP50'][name] = round(float(p.mean()), 4) if p.size else 0.0
    print(f"{label}: mAP@0.5={metrics['mAP_50']}  mAP@0.5:0.95={metrics['mAP_50_95']}")
    return metrics

rfdetr_metrics = score(rfdetr_dts, 'RF-DETR cascade')
sam3_metrics = score(sam3_dts, 'SAM-3')


## 7. Side-by-side comparison + verdict

In [ ]:
rows = []
print(f"{'class':18s} {'RF-DETR AP50':>14s} {'SAM-3 AP50':>14s}  gap?")
for name in CAT_ID:
    r = rfdetr_metrics['per_class_AP50'][name]
    s = sam3_metrics['per_class_AP50'][name]
    gap = ' <-- domain gap' if name in GAP_CATS else ''
    rows.append((name, r, s))
    print(f'{name:18s} {r:>14.4f} {s:>14.4f}{gap}')

common = [n for n in CAT_ID if n not in GAP_CATS]
common_rf = sum(rfdetr_metrics['per_class_AP50'][n] for n in common) / len(common)
common_sam = sum(sam3_metrics['per_class_AP50'][n] for n in common) / len(common)
gap_sam = sum(sam3_metrics['per_class_AP50'][n] for n in GAP_CATS) / len(GAP_CATS)

print(f"\nOverall mAP@0.5      RF-DETR={rfdetr_metrics['mAP_50']:.4f}   SAM-3={sam3_metrics['mAP_50']:.4f}")
print(f'Common-class avg AP  RF-DETR={common_rf:.4f}   SAM-3={common_sam:.4f}')
print(f'Domain-gap avg AP (RF-DETR is structurally ~0)  SAM-3={gap_sam:.4f}')
print(f'\nLatency  RF-DETR={len(samples)/rfdetr_seconds:.2f} img/s (2 passes/img)   '
      f'SAM-3={len(samples)/sam3_seconds:.2f} img/s ({n_concepts} passes/img)')

if sam3_metrics['mAP_50'] > rfdetr_metrics['mAP_50'] and sam3_seconds < rfdetr_seconds * 3:
    verdict = 'REPLACE -- SAM-3 beats the documented cascade broadly at acceptable latency cost.'
elif gap_sam > 0.1 and common_sam < common_rf:
    verdict = 'HYBRID -- SAM-3 wins the domain-gap classes only; keep RF-DETR for the bulk/fast common classes.'
else:
    verdict = 'KEEP RF-DETR -- documented cascade remains the better primary detector as-is.'
print('\nVERDICT:', verdict)

with open(OUT_DIR / 'SUMMARY.md', 'w') as f:
    f.write('# SAM-3 vs RF-DETR cascade -- head-to-head\n\n')
    f.write(f'- Images: {len(samples)}\n')
    f.write(f"- RF-DETR mAP@0.5: {rfdetr_metrics['mAP_50']} | SAM-3 mAP@0.5: {sam3_metrics['mAP_50']}\n")
    f.write(f'- RF-DETR img/s: {len(samples)/rfdetr_seconds:.2f} | SAM-3 img/s: {len(samples)/sam3_seconds:.2f}\n\n')
    f.write('| class | RF-DETR AP@0.5 | SAM-3 AP@0.5 |\n|---|---|---|\n')
    for name, r, s in rows:
        f.write(f'| {name} | {r:.4f} | {s:.4f} |\n')
    f.write(f'\n**Verdict:** {verdict}\n')
print('\nSaved ->', OUT_DIR / 'SUMMARY.md')


## How to read this
- **Common classes** (person/bicycle/car/motorcycle/bus/truck/traffic-light): both systems are
  COCO-trainable, so this is the fair apples-to-apples comparison.
- **Domain-gap classes** (auto-rickshaw/animal/traffic-sign/vehicle-fallback): RF-DETR is
  structurally ~0 here zero-shot (no COCO equivalent) -- SAM-3's score is the entire signal.
- **Latency**: SAM-3 cost scales linearly with the number of concepts queried per image; RF-DETR
  does not. If a hybrid wins, the natural design is SAM-3 as an **auto-labeler** (run once
  offline, distill into a fine-tuned RF-DETR for the gap classes) rather than an online per-frame
  dependency -- keeps the "scalable image analysis system" requirement intact either way.
